# Part 5 — Text Analyst

This notebook documents the workflow for the Text Analyst component of the Multimodal Crime / Incident Report Analyzer project.

The goal of this component is to process unstructured crime-related social/news text from the Kaggle CrimeReport dataset and convert it into structured incident information.

This notebook demonstrates:

1. Loading the raw CrimeReport JSON Lines dataset
2. Inspecting the original text records
3. Running the Part 5 NLP pipeline
4. Reviewing the Part 5 text-analysis output
5. Reviewing the team integration output
6. Checking topic and sentiment summaries

In [1]:
from pathlib import Path
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    """
    Find the project root by walking upward until the repository structure is found.
    """
    current = start_path.resolve()

    for path in [current] + list(current.parents):
        if (path / "README.md").exists() and (path / "text").exists():
            return path

    raise FileNotFoundError("Could not find project root. Please check the notebook location.")

PROJECT_ROOT = find_project_root(Path.cwd())

DATA_PATH = PROJECT_ROOT / "text" / "data" / "CrimeReport (1).txt"
TEXT_OUTPUT_PATH = PROJECT_ROOT / "text" / "outputs" / "part5_text_analysis_output.csv"
INTEGRATION_OUTPUT_PATH = PROJECT_ROOT / "text" / "outputs" / "part5_text_integration_output.csv"

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Raw data exists:", DATA_PATH.exists())
print("Text output exists:", TEXT_OUTPUT_PATH.exists())
print("Integration output exists:", INTEGRATION_OUTPUT_PATH.exists())

Current working directory: C:\Users\17982\Desktop\EDS 6397\Group 1\EDS6397-Group01-Incident-Analysis\text\notebooks
Project root: C:\Users\17982\Desktop\EDS 6397\Group 1\EDS6397-Group01-Incident-Analysis
Raw data exists: True
Text output exists: True
Integration output exists: True


## 1. Load and Inspect the Raw CrimeReport Dataset

The downloaded Kaggle CrimeReport file is a JSON Lines `.txt` file. Each line is one social/news-style text record related to crime or police activity.

Although the assignment description mentions a CSV file, the actual downloaded dataset is read with `pandas.read_json(..., lines=True)`.

In [2]:
raw_df = pd.read_json(DATA_PATH, lines=True)

print("Raw dataset shape:", raw_df.shape)
print("Columns:")
print(raw_df.columns.tolist())

raw_df[["id", "text", "created_at", "source"]].head()

Raw dataset shape: (115, 20)
Columns:
['lang', 'favorited', 'truncated', 'text', 'created_at', 'retweeted', 'source', 'place', 'user', 'retweet_count', 'id', 'favorite_count', 'urls', 'user_mentions', 'hashtags', 'retweeted_status', 'in_reply_to_screen_name', 'in_reply_to_user_id', 'possibly_sensitive', 'in_reply_to_status_id']


,id,text,created_at,source
0,429129916446031872,Active crime scene on I-59/20 near Jeff/Tusc C...,2014-01-31 05:51:59+00:00,"<a href=""http://tapbots.com/software/tweetbot/..."
1,429117247307923456,Police have arrested a suspect in the Monday s...,2014-01-31 05:01:39+00:00,"<a href=""http://www.hootsuite.com"" rel=""nofoll..."
2,429315798893461505,Lawsuit alleges Chicago Police strip-searched ...,2014-01-31 18:10:37+00:00,"<a href=""http://twitter.com/tweetbutton"" rel=""..."
3,429079910091476992,New NRA Advice: Don't Cooperate With Police If...,2014-01-31 02:33:17+00:00,web
4,429030956888899584,"Police failing to stamp out ‘honour crimes’, f...",2014-01-30 23:18:45+00:00,"<a href=""https://about.twitter.com/products/tw..."


## 2. Text Preprocessing Example

The Part 5 script cleans each raw text record by removing URLs, user mentions, hashtag symbols, and extra whitespace. It also tokenizes cleaned text with NLTK and removes common English stopwords.

The original raw text is preserved in the final output so the structured result can still be traced back to the original message.

In [3]:
import sys

SRC_PATH = PROJECT_ROOT / "text" / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from text_analyzer import clean_text, tokenize_text  # type: ignore

sample_text = raw_df.loc[0, "text"]
cleaned_text = clean_text(sample_text)
tokens = tokenize_text(cleaned_text)

print("Raw text:")
print(sample_text)

print("\nCleaned text:")
print(cleaned_text)

print("\nTokens after stopword removal:")
print(tokens[:30])

Raw text:
Active crime scene on I-59/20 near Jeff/Tusc Co line. One dead, one injured; shooting involved. Police search in the area; traffic stopped

Cleaned text:
Active crime scene on I-59/20 near Jeff/Tusc Co line. One dead, one injured; shooting involved. Police search in the area; traffic stopped

Tokens after stopword removal:
['active', 'crime', 'scene', 'i-59/20', 'near', 'jeff/tusc', 'co', 'line.', 'one', 'dead', 'one', 'injured', 'shooting', 'involved.', 'police', 'search', 'area', 'traffic', 'stopped']


## 3. Named Entity Recognition Example

The Part 5 pipeline uses spaCy Named Entity Recognition (NER) to extract useful entities from the cleaned text.

The selected entity types include:

- `PERSON` for people
- `GPE`, `LOC`, and `FAC` for locations and facilities
- `ORG` for organizations
- `DATE` for dates

In [4]:
import spacy
from text_analyzer import extract_entities  # type: ignore

nlp_model = spacy.load("en_core_web_sm")

sample_for_ner = raw_df.loc[1, "text"]
cleaned_for_ner = clean_text(sample_for_ner)

entities_string, entities_detail = extract_entities(cleaned_for_ner, nlp_model)

print("Sample text:")
print(cleaned_for_ner)

print("\nExtracted entities:")
print(entities_string)

print("\nDetailed entity list:")
print(entities_detail)

Sample text:
Police have arrested a suspect in the Monday shooting at the basketball courts behind Word Tabernacle Church.

Extracted entities:
Monday (DATE); Word Tabernacle Church (ORG)

Detailed entity list:
[{'text': 'Monday', 'label': 'DATE'}, {'text': 'Word Tabernacle Church', 'label': 'ORG'}]


## 4. Part 5 Text Analysis Output

The script generates the required Part 5 structured CSV with the following columns:

`Text_ID, Source, Raw_Text, Sentiment, Entities, Topic`

This output directly satisfies the Text Analyst component requirement.

In [5]:
text_output_df = pd.read_csv(TEXT_OUTPUT_PATH)

print("Part 5 text output shape:", text_output_df.shape)
print("Columns:", text_output_df.columns.tolist())

text_output_df.head()

Part 5 text output shape: (115, 6)
Columns: ['Text_ID', 'Source', 'Raw_Text', 'Sentiment', 'Entities', 'Topic']


,Text_ID,Source,Raw_Text,Sentiment,Entities,Topic
0,TXT-000001,Tweetbot for Mac,Active crime scene on I-59/20 near Jeff/Tusc C...,Negative,No entities found,Shooting
1,TXT-000002,HootSuite,Police have arrested a suspect in the Monday s...,Negative,Monday (DATE); Word Tabernacle Church (ORG),Shooting
2,TXT-000003,Tweet Button,Lawsuit alleges Chicago Police strip-searched ...,Negative,Chicago (GPE),Police Activity
3,TXT-000004,web,New NRA Advice: Don't Cooperate With Police If...,Negative,New NRA Advice (ORG),Police Activity
4,TXT-000005,TweetDeck,"Police failing to stamp out ‘honour crimes’, f...",Negative,FGM (ORG),Police Activity


## 5. Team Integration Output

For the final team integration, this component also generates a second CSV using the shared common output format:

`Incident_ID, Event_Type, Source_Type, Date_Time, Location, Confidence_Score, Summary`

The `Incident_ID` follows the group format, such as:

`INC-2026-004000`

The `Source_Type` is set to `Text` for every row.

In [6]:
integration_output_df = pd.read_csv(INTEGRATION_OUTPUT_PATH)

print("Integration output shape:", integration_output_df.shape)
print("Columns:", integration_output_df.columns.tolist())

integration_output_df.head()

Integration output shape: (115, 7)
Columns: ['Incident_ID', 'Event_Type', 'Source_Type', 'Date_Time', 'Location', 'Confidence_Score', 'Summary']


,Incident_ID,Event_Type,Source_Type,Date_Time,Location,Confidence_Score,Summary
0,INC-2026-004000,Shooting,Text,2014-01-31 05:51:59+00:00,"Hoover, AL",0.8143,Active crime scene on I-59/20 near Jeff/Tusc C...
1,INC-2026-004001,Shooting,Text,2014-01-31 05:01:39+00:00,"Rocky Mount, N.C.",0.3226,Police have arrested a suspect in the Monday s...
2,INC-2026-004002,Police Activity,Text,2014-01-31 18:10:37+00:00,Chicago,0.5889,Lawsuit alleges Chicago Police strip-searched ...
3,INC-2026-004003,Police Activity,Text,2014-01-31 02:33:17+00:00,Unknown,0.7984,New NRA Advice: Don't Cooperate With Police If...
4,INC-2026-004004,Police Activity,Text,2014-01-30 23:18:45+00:00,London,0.4470,"Police failing to stamp out ‘honour crimes’, f..."


## 6. Topic and Sentiment Summary

The generated Part 5 output can be summarized by topic and sentiment. This helps the final dashboard or query system understand the distribution of incident-related text records.

In [7]:
topic_summary = (
    text_output_df["Topic"]
    .value_counts()
    .rename_axis("Topic")
    .reset_index(name="Count")
)

sentiment_summary = (
    text_output_df["Sentiment"]
    .value_counts()
    .rename_axis("Sentiment")
    .reset_index(name="Count")
)

print("Topic counts:")
display(topic_summary)

print("Sentiment counts:")
display(sentiment_summary)

Topic counts:


,Topic,Count
0,Police Activity,97
1,Legal / Court,6
2,Other,5
3,Public Disturbance,3
4,Shooting,2
5,Theft / Robbery,2


Sentiment counts:


,Sentiment,Count
0,Negative,101
1,Positive,14


## 7. Output Validation

Before uploading to GitHub, the generated CSV outputs are validated to confirm that they follow both the Part 5 text-analysis format and the team integration format.

In [8]:
expected_text_columns = ["Text_ID", "Source", "Raw_Text", "Sentiment", "Entities", "Topic"]
expected_integration_columns = [
    "Incident_ID",
    "Event_Type",
    "Source_Type",
    "Date_Time",
    "Location",
    "Confidence_Score",
    "Summary",
]

assert list(text_output_df.columns) == expected_text_columns
assert list(integration_output_df.columns) == expected_integration_columns
assert len(text_output_df) == 115
assert len(integration_output_df) == 115
assert integration_output_df["Incident_ID"].str.match(r"^INC-2026-\d{6}$").all()
assert (integration_output_df["Source_Type"] == "Text").all()
assert integration_output_df["Confidence_Score"].between(0, 1).all()

print("Notebook validation passed.")
print("Part 5 text output:", text_output_df.shape)
print("Integration output:", integration_output_df.shape)

Notebook validation passed.
Part 5 text output: (115, 6)
Integration output: (115, 7)
